# Module 11 -- Wednesday Session Demo
## CS 5045: Computation for the Data Sciences

**Topic:** SQL and Relational Data

This notebook walks through every major SQL clause type using an in-memory SQLite database.
All data is created inline -- no external files needed. Run cells from top to bottom.

**SQL clauses covered:** SELECT, FROM, WHERE, ORDER BY, GROUP BY, HAVING, INNER JOIN

**Operations covered:** Represent, Filter, Transform, Aggregate, Communicate

---

In [ ]:
# Cell 1 -- imports
# sqlite3 is part of the Python standard library -- no pip install needed.
import sqlite3
import pandas as pd

print('sqlite3 version:', sqlite3.sqlite_version)
print('pandas version:', pd.__version__)

In [ ]:
# Cell 2 -- REPRESENT: create an in-memory database and populate two tables
#
# ':memory:' means the database lives in RAM only.
# It is gone when this Python session ends -- perfect for demos.
# In production you pass a file path: sqlite3.connect('mydata.db')

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# --- Table 1: customers ---
cursor.execute('''
CREATE TABLE customers (
    customer_id   INTEGER PRIMARY KEY,
    name          TEXT,
    state         TEXT,
    total_spent   REAL,
    year          INTEGER
)
''')

customers_data = [
    (1,  'Alice Morales',    'CA', 720.50, 2023),
    (2,  'Ben Tran',         'TX', 340.00, 2023),
    (3,  'Carla Osei',       'CA', 580.25, 2023),
    (4,  'David Kim',        'WA', 210.75, 2022),
    (5,  'Eva Patel',        'CA', 930.00, 2023),
    (6,  'Frank Nguyen',     'TX', 415.60, 2022),
    (7,  'Grace Liu',        'NY', 645.90, 2023),
    (8,  'Hector Ruiz',      'CA', 110.00, 2023),
    (9,  'Iris Johnson',     'WA', 780.40, 2023),
    (10, 'James Okonkwo',    'NY', 255.00, 2022),
    (11, 'Karen Shapiro',    'CA', 860.15, 2023),
    (12, 'Luis Ferreira',    'TX', 530.80, 2023),
    (13, 'Maya Singh',       'NY', 670.00, 2023),
    (14, 'Nora Andersen',    'WA', 495.30, 2023),
    (15, 'Oscar Chen',       'CA', 305.00, 2022),
]
cursor.executemany('INSERT INTO customers VALUES (?, ?, ?, ?, ?)', customers_data)

# --- Table 2: orders (for the JOIN demonstration) ---
cursor.execute('''
CREATE TABLE orders (
    order_id      INTEGER PRIMARY KEY,
    customer_id   INTEGER,
    product       TEXT,
    amount        REAL
)
''')

orders_data = [
    (101, 1,  'Laptop Stand',    89.99),
    (102, 1,  'USB Hub',         34.50),
    (103, 3,  'Webcam',          129.00),
    (104, 5,  'Monitor',         399.00),
    (105, 5,  'Keyboard',        79.99),
    (106, 7,  'Mouse',           45.00),
    (107, 9,  'Headset',         120.00),
    (108, 11, 'Docking Station', 199.99),
    (109, 11, 'Cable Kit',       25.00),
    (110, 12, 'Desk Lamp',       65.00),
    (111, 13, 'Chair Mat',       55.00),
    (112, 14, 'Surge Protector', 39.99),
]
cursor.executemany('INSERT INTO orders VALUES (?, ?, ?, ?)', orders_data)

conn.commit()
print('Database created.')
print('Tables: customers (15 rows), orders (12 rows)')

In [ ]:
# Cell 3 -- SELECT all, LIMIT 5
#
# The most basic query: give me everything from a table, first five rows only.
# Compare to df.head() in pandas -- same idea, SQL syntax.
#
# Operation: Represent (retrieving data from the database)

query = '''
SELECT *
FROM   customers
LIMIT  5
'''

cursor.execute(query)
rows = cursor.fetchall()

print('Columns: customer_id, name, state, total_spent, year')
print()
for row in rows:
    print(row)

In [ ]:
# Cell 4 -- WHERE: Filter rows by condition
#
# WHERE is the SQL equivalent of boolean indexing in pandas.
#   SQL:    WHERE state = 'CA' AND total_spent > 500
#   pandas: df[(df['state'] == 'CA') & (df['total_spent'] > 500)]
#
# Operation: Filter

query = '''
SELECT name, state, total_spent, year
FROM   customers
WHERE  state = 'CA'
  AND  total_spent > 500
  AND  year = 2023
'''

cursor.execute(query)
rows = cursor.fetchall()

print('CA customers who spent > $500 in 2023:')
print()
for row in rows:
    name, state, spent, year = row
    print(f'  {name:<20} {state}  ${spent:>7.2f}  {year}')
print(f'\n{len(rows)} row(s) returned.')

In [ ]:
# Cell 5 -- ORDER BY: sort results
#
# ORDER BY is the SQL equivalent of df.sort_values().
# DESC = largest first. ASC = smallest first (the default).
#
# Operation: Transform (rearranging the output)

query = '''
SELECT name, state, total_spent
FROM   customers
WHERE  year = 2023
ORDER BY total_spent DESC
LIMIT  5
'''

cursor.execute(query)
rows = cursor.fetchall()

print('Top 5 spenders in 2023:')
print()
for rank, row in enumerate(rows, start=1):
    name, state, spent = row
    print(f'  {rank}. {name:<20} {state}  ${spent:>7.2f}')

In [ ]:
# Cell 6 -- GROUP BY + HAVING: aggregate per group, then filter groups
#
# GROUP BY is the SQL equivalent of .groupby() from Module 10.
# HAVING filters groups after aggregation -- WHERE cannot do this
# because WHERE runs before GROUP BY.
#
#   SQL:    GROUP BY state HAVING COUNT(*) >= 2
#   pandas: df.groupby('state')['total_spent'].agg(['count','mean'])
#           then filter rows where count >= 2
#
# Operation: Aggregate

query = '''
SELECT state,
       COUNT(*)           AS num_customers,
       ROUND(AVG(total_spent), 2) AS avg_spent
FROM   customers
WHERE  year = 2023
GROUP BY state
HAVING COUNT(*) >= 2
ORDER BY avg_spent DESC
'''

# Read directly into a DataFrame using pd.read_sql_query (preview of Cell 8)
result_df = pd.read_sql_query(query, conn)

print('States with 2 or more 2023 customers, average spend:')
print()
print(result_df.to_string(index=False))

In [ ]:
# Cell 7 -- INNER JOIN: combine two tables on a shared key
#
# INNER JOIN is the SQL equivalent of pd.merge(df1, df2, on='customer_id', how='inner').
# Only rows where customer_id exists in BOTH tables appear in the result.
# Customers with no orders are excluded.
#
# Operation: Represent (connecting related data)

query = '''
SELECT c.name,
       c.state,
       o.product,
       o.amount
FROM   customers AS c
INNER JOIN orders AS o
        ON c.customer_id = o.customer_id
ORDER BY c.name
'''

join_df = pd.read_sql_query(query, conn)

print('Customers with their individual orders (INNER JOIN):')
print()
print(join_df.to_string(index=False))
print(f'\n{len(join_df)} rows -- one per order, not one per customer.')
print('Customers with no orders do not appear (INNER JOIN excludes them).')

In [ ]:
# Cell 8 -- pd.read_sql_query(): the pandas shortcut
#
# In production work, you rarely call cursor.fetchall() by hand.
# pd.read_sql_query() runs the SQL and returns a DataFrame directly.
# The result is a normal pandas DataFrame -- you can chain any pandas operation onto it.
#
# Operation: Communicate (bringing results into a form ready for analysis or display)

query = '''
SELECT c.name,
       c.state,
       COUNT(o.order_id)         AS num_orders,
       ROUND(SUM(o.amount), 2)   AS total_order_value
FROM   customers AS c
INNER JOIN orders AS o
        ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name, c.state
ORDER BY total_order_value DESC
'''

summary = pd.read_sql_query(query, conn)

print('Total order value per customer (pd.read_sql_query result):')
print()
print(summary.to_string(index=False))
print()
print('dtypes after read_sql_query:')
print(summary.dtypes)
print()
print('The result is a regular pandas DataFrame.')
print('You can call .describe(), .plot(), .to_csv() -- anything you already know.')